# Lab 9 – Data Cleaning
**Dataset:** ArXiv AI-detection papers collected by the pipeline  
**Raw input:** `data/raw/csv/papers_raw.csv`  
**Cleaned output:** `data/processed/cleaned/cleaned_data.csv`

## Part 1 – Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

RAW_CSV = '../data/raw/csv/papers_raw.csv'
df_raw = pd.read_csv(RAW_CSV)
print(f'Loaded {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head()

## Part 2 – Missing Value Analysis

In [ ]:
from src.cleaning.missing_handler import report_missing

missing_report = report_missing(df_raw)
print(missing_report.to_string())

missing_report.to_csv('../data/processed/cleaned/missing_report.csv')
print('\nSaved to data/processed/cleaned/missing_report.csv')

In [ ]:
from src.cleaning.missing_handler import (
    drop_missing_ids,
    fill_missing_authors,
    fill_missing_abstracts,
    fill_missing_numeric,
    drop_high_missing_cols,
)

df = df_raw.copy()
print(f'Rows before: {len(df)}')

df = drop_high_missing_cols(df, threshold=0.5)
df = drop_missing_ids(df)
df = fill_missing_authors(df)
df = fill_missing_abstracts(df)
df = fill_missing_numeric(df)

print(f'Rows after:  {len(df)}')
print(f'Missing values remaining:\n{df.isnull().sum().to_string()}')

## Part 3 – String Cleaning

In [ ]:
from src.cleaning.string_cleaner import (
    clean_titles,
    normalize_language,
    clean_abstract,
    extract_year,
    clean_categories,
)

df_str = df.copy()
df_str = clean_titles(df_str)
df_str = normalize_language(df_str)
df_str = clean_abstract(df_str)
df_str = clean_categories(df_str)
df_str = extract_year(df_str)

print('Language values after normalisation:', df_str['language'].unique())
print('\nTitle preview:')
print(df_str[['title', 'published_date', 'published_year', 'language', 'primary_category']].to_string())

## Part 4 – Regex Cleaning (complex patterns)

In [ ]:
import re

def is_valid_date(val: str) -> bool:
    """Return True if val matches YYYY-MM-DD."""
    return bool(re.fullmatch(r'\d{4}-\d{2}-\d{2}', str(val).strip()))

def is_valid_arxiv_id(val: str) -> bool:
    """Return True if val matches the ArXiv YYMM.NNNNN pattern."""
    return bool(re.fullmatch(r'\d{4}\.\d{4,5}', str(val).strip()))

def extract_numeric_from_text(text: str) -> float | None:
    """Extract the first number found in a string."""
    m = re.search(r'[\d]+\.?[\d]*', str(text))
    return float(m.group()) if m else None

def flag_short_abstract(text: str, min_words: int = 10) -> bool:
    """Return True if abstract has fewer than min_words words."""
    return len(str(text).split()) < min_words

# Apply to raw data to flag suspicious records
print('Invalid date formats:')
bad_dates = df_raw[~df_raw['published_date'].astype(str).apply(is_valid_date)]
print(bad_dates[['paper_id', 'published_date']])

print('\nInvalid ArXiv IDs:')
bad_ids = df_raw[df_raw['paper_id'].notna() & ~df_raw['paper_id'].astype(str).apply(is_valid_arxiv_id)]
print(bad_ids[['paper_id']])

print('\nShort abstracts (< 10 words):')
short = df_raw[df_raw['abstract'].fillna('').apply(flag_short_abstract)]
print(short[['paper_id', 'abstract']])

## Part 5 – Deduplication

In [ ]:
from src.cleaning.deduplicator import count_duplicates, drop_exact_duplicates, drop_key_duplicates

df_dedup = df_str.copy()
print(f'Rows before deduplication: {len(df_dedup)}')
print(f'Exact duplicates: {count_duplicates(df_dedup)}')
print(f'Duplicate paper_ids: {count_duplicates(df_dedup, col="paper_id")}')

df_dedup = drop_exact_duplicates(df_dedup)
print(f'\nAfter dropping exact duplicates: {len(df_dedup)} rows')

print(f'Duplicate paper_ids before key-dedup: {count_duplicates(df_dedup, col="paper_id")}')
df_dedup = drop_key_duplicates(df_dedup, key='paper_id')
print(f'After dropping key duplicates: {len(df_dedup)} rows')

## Part 6 – Type Conversion

In [ ]:
from src.cleaning.type_converter import convert_dates, convert_numerics, convert_categories, memory_report

df_before_types = df_dedup.copy()

df_typed = convert_dates(df_dedup.copy())
df_typed = convert_numerics(df_typed)
df_typed = convert_categories(df_typed)

print('Updated dtypes:')
print(df_typed.dtypes.to_string())
print()
print('Memory usage report:')
print(memory_report(df_before_types, df_typed).to_string())

## Part 7 – Validation

In [ ]:
from src.cleaning.validator import run_all_validations

# Clamp any out-of-range relevance scores before final validation
df_typed['relevance_score'] = df_typed['relevance_score'].clip(0.0, 1.0)

results = run_all_validations(df_typed)
for check, outcome in results.items():
    status = '✓' if outcome == 'PASSED' else '✗'
    print(f'{status} {check}: {outcome}')

## Part 8 – Full Cleaning Pipeline

In [ ]:
from src.cleaning.clean_pipeline import run_cleaning_pipeline
from pathlib import Path

cleaned = run_cleaning_pipeline(
    raw_path=Path('../data/raw/csv/papers_raw.csv'),
    output_path=Path('../data/processed/cleaned/cleaned_data.csv'),
)

print(f'\nFinal cleaned dataset: {cleaned.shape[0]} rows × {cleaned.shape[1]} columns')
cleaned.head()

## Part 9 – Run Tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', '../tests/test_cleaning.py', '-v'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)